# Notebook 5 — Final Comparison & Analysis
**All Team Members**

This notebook loads saved results from all experiments and produces:
1. Side-by-side comparison table
2. Bar chart comparison
3. Analysis discussion points for the report

In [ ]:
#@title 1. Setup
from pathlib import Path
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/wild_animal_image_classification'),
    Path('/content'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'configs' / 'config.py').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from configs.config import CFG

CFG.ensure_output_dirs()
RESULTS_DIR = CFG.results_dir

In [ ]:
#@title 2. Load All Results
result_files = {
    'EfficientNet-B4': 'efficientnet_b4_final.json',
    'ViT-B/16': 'vit_b16_final.json',
    'DINOv2 Linear Probe': 'dinov2_linear_probe_final.json',
    'DINOv2 Fine-tune': 'dinov2_finetune_final.json',
}

all_results = {}
for name, fname in result_files.items():
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        with open(path) as f:
            all_results[name] = json.load(f)
        print(f'Loaded: {name}')
    else:
        print(f'MISSING: {name} ({fname})')

# Clustering
cluster_path = f'{RESULTS_DIR}/clustering_results.json'
if os.path.exists(cluster_path):
    with open(cluster_path) as f:
        cluster_results = json.load(f)
    print('Loaded: Clustering results')
else:
    cluster_results = None
    print('MISSING: Clustering results')

In [ ]:
#@title 3. Comparison Table
rows = []
for name, r in all_results.items():
    rows.append({
        'Model': name,
        'Accuracy': f"{r['accuracy']:.4f}",
        'F1 (macro)': f"{r['f1_macro']:.4f}",
        'F1 (weighted)': f"{r['f1_weighted']:.4f}",
        'Category': r.get('category', ''),
    })

comparison_df = pd.DataFrame(rows)
print('\n' + '='*70)
print('  CLASSIFICATION RESULTS COMPARISON')
print('='*70)
print(comparison_df.to_string(index=False))
print('='*70)

if cluster_results:
    print(f'\n  CLUSTERING (K={cluster_results.get("best_k", "?")}):')
    for k, v in cluster_results.items():
        if isinstance(v, dict):
            print(f'    K={k}: NMI={v["nmi"]:.4f}, ARI={v["ari"]:.4f}, Sil={v["silhouette"]:.4f}')
    if 'cluster_accuracy_hungarian' in cluster_results:
        print(f'  Cluster Accuracy (Hungarian): {cluster_results["cluster_accuracy_hungarian"]:.4f}')

In [ ]:
#@title 4. Comparison Bar Chart
names = list(all_results.keys())
acc_vals = [all_results[n]['accuracy'] for n in names]
f1_vals = [all_results[n]['f1_macro'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, acc_vals, width, label='Accuracy', color='#4C72B0')
bars2 = ax.bar(x + width/2, f1_vals, width, label='F1 (macro)', color='#DD8452')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — iWildCam 2019 (Empty Removed)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/model_comparison.png')

In [ ]:
#@title 5. Analysis Discussion Points
print("""
Discussion points for the report and presentation
------------------------------------------------

1. CNN vs Transformer:
   - How does EfficientNet-B4 compare to ViT-B/16?
   - Which handles the class imbalance better?

2. Self-Supervised Features (DINOv2):
   - How well does a linear probe on frozen features perform
     compared to full fine-tuning?
   - Does fine-tuning the last 4 blocks close the gap?

3. Unsupervised Clustering:
   - Do the DINOv2 clusters align with species boundaries?
   - What does the NMI/ARI tell us about feature quality?
   - Do UMAP/t-SNE show clear species separation?

4. Class Imbalance:
   - Which species are hardest to classify across all models?
   - Does the weighted sampler help with rare classes?

5. Practical Insights:
   - Which model gives the best accuracy-to-compute tradeoff?
   - How might these results help real wildlife monitoring?
------------------------------------------------
""")